In [ ]:
# =============================================================================
# Model 2 (Retention Intelligence LLM) — Train BOTH sizes, evaluate on your
# held-out test.jsonl, keep both merged models, export GGUF for both, and
# clearly flag the winner. Runs on a free Colab T4 GPU.
# =============================================================================
# Runtime > Change runtime type > T4 GPU, then run cells top to bottom.
# Upload train.jsonl, val.jsonl, test.jsonl to the Colab session first
# (or mount Google Drive and point the paths at your Drive folder).
# =============================================================================

# --- install deps ---------------------------------------------------
!pip install -q transformers peft bitsandbytes trl accelerate datasets



In [ ]:
import gc
import json
import os
import random
import time

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
random.seed(42)

TRAIN_PATH = "/content/train_modified.jsonl"
VAL_PATH = "/content/val_modified.jsonl"
TEST_PATH = "/content/test_modified.jsonl"

MODEL_CANDIDATES = [
    {"name": "Qwen/Qwen2.5-0.5B-Instruct", "tag": "0.5b", "out_dir": "model2-0.5b-lora"},
]

ALLOWED_PREFIXES = {"rm_call", "rate_offer", "fee_waiver", "complaint_escalation", "do_nothing"}

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        records = []
        for l in f:
            rec = json.loads(l)
            # Ensure 'messages' and its content are consistently typed
            if isinstance(rec.get("messages"), list):
                cleaned_messages = []
                for msg in rec["messages"]:
                    if isinstance(msg, dict):
                        cleaned_msg = msg.copy()
                        if "role" in cleaned_msg and not isinstance(cleaned_msg["role"], str):
                            cleaned_msg["role"] = str(cleaned_msg["role"])
                        if "content" in cleaned_msg and not isinstance(cleaned_msg["content"], str):
                            # Convert non-string content (including None) to string
                            cleaned_msg["content"] = str(cleaned_msg["content"] or "")
                        cleaned_messages.append(cleaned_msg)
                rec["messages"] = cleaned_messages
            else:
                # If messages is not a list, normalize it to an empty list or handle as appropriate
                # For now, let's just ensure it's a list if it's there.
                rec["messages"] = [] # Or you could choose to skip this record if 'messages' is critical

            records.append(rec)
        return records


# The is_valid_messages_list is now less critical for type consistency due to pre-processing in load_jsonl
# but can still be used for semantic validation if needed.
def is_valid_messages_list(messages):
    """Checks if 'messages' is a list and all its elements are dictionaries."""
    return isinstance(messages, list) and all(isinstance(msg, dict) for msg in messages)


train_records = load_jsonl(TRAIN_PATH)
val_records = load_jsonl(VAL_PATH)
test_records = load_jsonl(TEST_PATH)
print(f"Train: {len(train_records)}  Val: {len(val_records)}  Test: {len(test_records)}")

# Filter out malformed records to ensure consistent schema for datasets
# The 'messages' key is crucial for subsequent operations.
# While load_jsonl now cleans, this filter ensures structural integrity.
train_records = [rec for rec in train_records if is_valid_messages_list(rec.get("messages"))]
val_records = [rec for rec in val_records if is_valid_messages_list(rec.get("messages"))]
print(f"Filtered Train: {len(train_records)}  Filtered Val: {len(val_records)}")

train_ds = Dataset.from_list(train_records)
val_ds = Dataset.from_list(val_records)

Train: 960  Val: 120  Test: 120
Filtered Train: 960  Filtered Val: 120


In [ ]:
# --- training function (reused per model) ---------------------------
def train_one_model(model_name, out_dir):
    print(f"\n{'='*70}\nTraining {model_name}\n{'='*70}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    def format_example(example):
        text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
        return {"text": text}

    train_fmt = train_ds.map(format_example)
    val_fmt = val_ds.map(format_example)

    sft_config = SFTConfig(
        output_dir=out_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        bf16=True,
        max_length=1024,
        dataset_text_field="text",
        report_to="none",
    )

    trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_fmt, eval_dataset=val_fmt)
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    return model, tokenizer

In [ ]:
# --- evaluation function (reused per model) --------------------------
def extract_json_numbers(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for v in obj.values():
            extract_json_numbers(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            extract_json_numbers(v, acc)
    elif isinstance(obj, (int, float)):
        acc.add(abs(round(obj)))
    return acc


def extract_text_numbers(text_list):
    import re
    nums = set()
    for t in text_list:
        for m in re.findall(r"-?\d+\.?\d*", t):
            try:
                nums.add(abs(round(float(m))))
            except ValueError:
                pass
    return nums

In [ ]:
def evaluate_model(model, tokenizer, test_records, max_new_tokens=300):
    model.eval()
    n = len(test_records)
    json_valid = 0
    prefixes_valid = 0
    grounded_checked = 0
    grounded_ok = 0
    latencies = []

    for rec in test_records:
        system_msg, user_msg, _ = rec["messages"]
        prompt = tokenizer.apply_chat_template(
            [system_msg, user_msg], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        t0 = time.time()
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        latencies.append(time.time() - t0)

        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        try:
            gen = json.loads(completion)
            assert "why" in gen and "next_actions" in gen and isinstance(gen["why"], list) and isinstance(gen["next_actions"], list)
            json_valid += 1
        except Exception:
            continue  # can't score prefix/grounding on invalid JSON

        prefixes = [a.split(":", 1)[0].strip() for a in gen["next_actions"] if ":" in a]
        if prefixes and all(p in ALLOWED_PREFIXES for p in prefixes):
            prefixes_valid += 1

        user_obj = json.loads(user_msg["content"])
        json_nums = extract_json_numbers(user_obj)
        why_nums = extract_text_numbers(gen["why"])
        if why_nums:
            grounded_checked += 1
            if why_nums & json_nums:
                grounded_ok += 1

    json_valid_rate = json_valid / n
    prefix_valid_rate = prefixes_valid / n
    grounding_rate = (grounded_ok / grounded_checked) if grounded_checked else None
    avg_latency = sum(latencies) / len(latencies)

    # composite: JSON validity and correct action vocabulary matter most for
    # a production dashboard integration; grounding is a secondary quality signal.
    composite = 0.5 * json_valid_rate + 0.3 * prefix_valid_rate + 0.2 * (grounding_rate or 0)

    return {
        "json_valid_rate": round(json_valid_rate, 3),
        "prefix_valid_rate": round(prefix_valid_rate, 3),
        "grounding_rate": round(grounding_rate, 3) if grounding_rate is not None else None,
        "avg_latency_sec": round(avg_latency, 2),
        "composite_score": round(composite, 3),
    }

In [ ]:
# --- run both models end-to-end ------------------------------------
results = {}
trained_models = {}  # keep in memory only long enough to eval + merge, then free

for cfg in MODEL_CANDIDATES:
    model, tokenizer = train_one_model(cfg["name"], cfg["out_dir"])
    metrics = evaluate_model(model, tokenizer, test_records)
    metrics["out_dir"] = cfg["out_dir"]
    metrics["base_model"] = cfg["name"]
    results[cfg["tag"]] = metrics
    print(f"\n{cfg['tag']} results: {json.dumps(metrics, indent=2)}")

    # free GPU memory before loading the next candidate
    del model
    gc.collect()
    torch.cuda.empty_cache()


Training Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Truncating eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.131906,0.133677,0.135887,264246.000000,0.948382
2,0.126652,0.127928,0.129615,528492.000000,0.949171
3,0.120634,0.124966,0.127573,792738.000000,0.950712



0.5b results: {
  "json_valid_rate": 0.0,
  "prefix_valid_rate": 0.0,
  "grounding_rate": null,
  "avg_latency_sec": 8.17,
  "composite_score": 0.0,
  "out_dir": "model2-0.5b-lora",
  "base_model": "Qwen/Qwen2.5-0.5B-Instruct"
}


In [ ]:
!zip -r /content/my_archive.zip /content/model2-0.5b-lora/
# !unzip my_archive.zip

  adding: content/model2-0.5b-lora/ (stored 0%)
  adding: content/model2-0.5b-lora/checkpoint-60/ (stored 0%)
  adding: content/model2-0.5b-lora/checkpoint-60/trainer_state.json (deflated 69%)
  adding: content/model2-0.5b-lora/checkpoint-60/adapter_model.safetensors (deflated 21%)
  adding: content/model2-0.5b-lora/checkpoint-60/tokenizer_config.json (deflated 59%)
  adding: content/model2-0.5b-lora/checkpoint-60/training_args.bin (deflated 53%)
  adding: content/model2-0.5b-lora/checkpoint-60/adapter_config.json (deflated 60%)
  adding: content/model2-0.5b-lora/checkpoint-60/chat_template.jinja (deflated 71%)
  adding: content/model2-0.5b-lora/checkpoint-60/README.md (deflated 65%)
  adding: content/model2-0.5b-lora/checkpoint-60/rng_state.pth (deflated 26%)
  adding: content/model2-0.5b-lora/checkpoint-60/tokenizer.json (deflated 81%)
  adding: content/model2-0.5b-lora/checkpoint-60/scheduler.pt (deflated 61%)
  adding: content/model2-0.5b-lora/checkpoint-60/optimizer.pt (deflated 2

In [ ]:
!find /content -iname "adapter_config.json"

/content/model2-0.5b-lora/checkpoint-60/adapter_config.json
/content/model2-0.5b-lora/adapter_config.json
/content/model2-0.5b-lora/checkpoint-180/adapter_config.json
/content/model2-0.5b-lora/checkpoint-120/adapter_config.json


In [ ]:
!pip install --upgrade torchao
!pip install -q transformers peft accelerate

# --- merge LoRA adapters into base weights (for BOTH models) --------
def merge_and_save(base_model_name, adapter_dir, merged_dir):
    print(f"\nMerging {base_model_name} + {adapter_dir} -> {merged_dir}")
    base = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.bfloat16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
    merged.save_pretrained(merged_dir)
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    tokenizer.save_pretrained(merged_dir)
    del base, merged
    gc.collect()
    torch.cuda.empty_cache()


for cfg in MODEL_CANDIDATES:
    merged_dir = f"merged_{cfg['tag']}"
    merge_and_save(cfg["name"], cfg["out_dir"], merged_dir)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Merging Qwen/Qwen2.5-0.5B-Instruct + model2-0.5b-lora -> merged_0.5b


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# --- convert both merged models to GGUF (Q4_K_M) --------------------
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

Cloning into 'llama.cpp'...
remote: Enumerating objects: 3855, done.
remote: Counting objects: 100% (3855/3855), done.
remote: Compressing objects: 100% (3144/3144), done.
remote: Total 3855 (delta 688), reused 2688 (delta 629), pack-reused 0 (from 0)
Receiving objects: 100% (3855/3855), 35.55 MiB | 6.32 MiB/s, done.
Resolving deltas: 100% (688/688), done.
Updating files: 100% (3505/3505), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 29.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 98.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━

In [ ]:
!cmake -B llama.cpp/build llama.cpp

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.3.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [ ]:
!cmake --build llama.cpp/build --config Release -j 2

[  0%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/hash.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  1%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/xxhash/xxhash.c.o
[  1%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/sha1/sha1.c.o
[  1%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/sha256/sha256.c.o
[  2%] Linking CXX static library libvendor-hash.a
[  2%] Built target vendor-hash
[  2%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-ba

In [ ]:
import json

path = "merged_0.5b/tokenizer_config.json"
with open(path) as f:
    cfg = json.load(f)

if isinstance(cfg.get("extra_special_tokens"), list):
    print(f"Found list: {cfg['extra_special_tokens']} — converting to empty dict")
    cfg["extra_special_tokens"] = {}
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2)
    print("Fixed.")
else:
    print(f"extra_special_tokens is already {type(cfg.get('extra_special_tokens'))} — this isn't it, check again.")

Found list: ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>'] — converting to empty dict
Fixed.


In [ ]:
!python llama.cpp/convert_hf_to_gguf.py merged_0.5b --outtype f16 --outfile model2_retention_0.5b_f16.gguf

INFO:hf-to-gguf:Loading model: merged_0.5b
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {896, 128}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.bfloat16 -

In [ ]:
!./llama.cpp/build/bin/llama-quantize model2_retention_0.5b_f16.gguf model2_retention_0.5b.gguf Q4_K_M

version: 0.3.0-dev (build 1, commit 3173a56)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing 'model2_retention_0.5b_f16.gguf' to 'model2_retention_0.5b.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 27 key-value pairs and 290 tensors from model2_retention_0.5b_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:            general.sampling.penalty_repeat f32       

In [ ]:
import os
size_mb = os.path.getsize("model2_retention_0.5b.gguf") / (1024 * 1024)
print(f"{size_mb:.1f} MB")

379.4 MB


In [ ]:
from google.colab import files
files.download("model2_retention_0.5b.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Zip all files in the current working directory (/content)
!zip -r workspace_files.zip . -x ".*"

# 2. Download the zip file to your local computer
from google.colab import files
files.download('workspace_files.zip')


  adding: sample_data/ (stored 0%)
  adding: sample_data/README.md (deflated 39%)
  adding: sample_data/anscombe.json (deflated 83%)
  adding: sample_data/california_housing_test.csv (deflated 76%)
  adding: sample_data/mnist_train_small.csv (deflated 88%)
  adding: sample_data/california_housing_train.csv (deflated 79%)
  adding: sample_data/mnist_test.csv (deflated 88%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>